In [16]:
import uuid
import json
import re
from typing import List
from datetime import datetime
from aisuite import Client

In [17]:
def planner_agent(topic: str, model: str = "openai:o4-mini") -> List[str]:
    """
    Generates a step-by-step research plan for a given topic.
    
    This function uses an AI model to break down a research task into actionable steps
    that will be executed by different agents (Research, Writer, Editor).
    
    Args:
        topic: The research topic/prompt provided by the user
        model: The AI model to use for planning (default: "openai:o4-mini")
    
    Returns:
        A list of strings, where each string is a step description to be executed
    """
    
    # Construct the prompt that instructs the AI to create a research plan
    prompt = f"""
        You are a planning agent responsible for organizing a research workflow using multiple intelligent agents.

        Available agents:
        - Research agent: MUST begin with a broad **web search using Tavily** to identify only **relevant** and **authoritative** items (e.g., high-impact venues, seminal works, surveys, or recent comprehensive sources). The output of this step MUST capture for each candidate: title, authors, year, venue/source, URL, and (if available) DOI.
        - Research agent: AFTER the Tavily step, perform a **targeted arXiv search** ONLY for the candidates discovered in the web step (match by title/author/DOI). If an arXiv preprint/version exists, record its arXiv URL and version info. Do NOT run a generic arXiv search detached from the Tavily results.
        - Writer agent: drafts based on research findings.
        - Editor agent: reviews, reflects on, and improves drafts.

        Produce a clear step-by-step research plan **as a valid Python list of strings** (no markdown, no explanations). 
        Each step must be atomic, actionable, and assigned to one of the agents.
        Maximum of 7 steps.

        DO NOT include steps like "create CSV", "set up repo", "install packages".
            - Focus on meaningful research tasks (search, extract, rank, draft, revise).
            - The FIRST step MUST be exactly: "Research agent: Use Tavily to perform a broad web search and collect top relevant items (title, authors, year, venue/source, URL, DOI if available)."
            - The SECOND step MUST be exactly: "Research agent: For each collected item, search on arXiv to find matching preprints/versions and record arXiv URLs (if they exist)."

        The FINAL step MUST instruct the writer agent to generate a comprehensive Markdown report that:
        - Uses all findings and outputs from previous steps
        - Includes inline citations (e.g., [1], (Wikipedia/arXiv))
        - Includes a References section with clickable links for all citations
        - Preserves earlier sources
        - Is detailed and self-contained

        Topic: "{topic}"
    """

    # Call the AI model to generate the plan
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,  # High temperature for more creative/diverse planning
    )

    # Extract the raw response text from the AI
    raw = response.choices[0].message.content.strip()
    return raw

In [18]:
def planner_agent(topic: str, model: str = "openai:o4-mini") -> List[str]:
    """
    Generates a step-by-step research plan for a given topic.
    
    This function uses an AI model to break down a research task into actionable steps
    that will be executed by different agents (Research, Writer, Editor).
    
    Args:
        topic: The research topic/prompt provided by the user
        model: The AI model to use for planning (default: "openai:o4-mini")
    
    Returns:
        A list of strings, where each string is a step description to be executed
    """
    
    # Construct the prompt that instructs the AI to create a research plan
    prompt = f"""
        You are a planning agent responsible for organizing a research workflow using multiple intelligent agents.

        Available agents:
        - Research agent: MUST begin with a broad **web search using Tavily** to identify only **relevant** and **authoritative** items (e.g., high-impact venues, seminal works, surveys, or recent comprehensive sources). The output of this step MUST capture for each candidate: title, authors, year, venue/source, URL, and (if available) DOI.
        - Research agent: AFTER the Tavily step, perform a **targeted arXiv search** ONLY for the candidates discovered in the web step (match by title/author/DOI). If an arXiv preprint/version exists, record its arXiv URL and version info. Do NOT run a generic arXiv search detached from the Tavily results.
        - Writer agent: drafts based on research findings.
        - Editor agent: reviews, reflects on, and improves drafts.

        Produce a clear step-by-step research plan **as a valid Python list of strings** (no markdown, no explanations). 
        Each step must be atomic, actionable, and assigned to one of the agents.
        Maximum of 7 steps.

        DO NOT include steps like "create CSV", "set up repo", "install packages".
            - Focus on meaningful research tasks (search, extract, rank, draft, revise).
            - The FIRST step MUST be exactly: "Research agent: Use Tavily to perform a broad web search and collect top relevant items (title, authors, year, venue/source, URL, DOI if available)."
            - The SECOND step MUST be exactly: "Research agent: For each collected item, search on arXiv to find matching preprints/versions and record arXiv URLs (if they exist)."

        The FINAL step MUST instruct the writer agent to generate a comprehensive Markdown report that:
        - Uses all findings and outputs from previous steps
        - Includes inline citations (e.g., [1], (Wikipedia/arXiv))
        - Includes a References section with clickable links for all citations
        - Preserves earlier sources
        - Is detailed and self-contained

        Topic: "{topic}"
    """

    # Call the AI model to generate the plan
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,  # High temperature for more creative/diverse planning
    )

    # Extract the raw response text from the AI
    raw = response.choices[0].message.content.strip()

   # --- Robust parsing: Try multiple methods to extract a list from the response ---
    def _coerce_to_list(s: str) -> List[str]:
        """
        Attempts to parse the AI response into a Python list of strings.
        Tries multiple parsing strategies in order of preference.
        """
        # Strategy 1: Try strict JSON parsing
        try:
            obj = json.loads(s)
            if isinstance(obj, list) and all(isinstance(x, str) for x in obj):
                return obj[:7]  # Limit to 7 steps as specified
        except json.JSONDecodeError:
            pass
        
        # Strategy 2: Try Python literal evaluation (safer than eval)
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list) and all(isinstance(x, str) for x in obj):
                return obj[:7]
        except Exception:
            pass
        
        # Strategy 3: If response is wrapped in code fences (```), extract and try again
        if s.startswith("```") and s.endswith("```"):
            inner = s.strip("`")
            try:
                obj = ast.literal_eval(inner)
                if isinstance(obj, list) and all(isinstance(x, str) for x in obj):
                    return obj[:7]
            except Exception:
                pass
        
        # If all parsing attempts fail, return empty list
        return []

    # Parse the raw response into a list of steps
    steps = _coerce_to_list(raw)

    # Define the required steps that must be present in the plan
    required_first = "Research agent: Use Tavily to perform a broad web search and collect top relevant items (title, authors, year, venue/source, URL, DOI if available)."
    required_second = "Research agent: For each collected item, search on arXiv to find matching preprints/versions and record arXiv URLs (if they exist)."
    final_required = "Writer agent: Generate the final comprehensive Markdown report with inline citations and a complete References section with clickable links."

    def _ensure_contract(steps_list: List[str]) -> List[str]:
        if not steps_list:
            return [
                required_first,
                required_second,
                "Research agent: Synthesize and rank findings by relevance, recency, and authority; deduplicate by title/DOI.",
                "Writer agent: Draft a structured outline based on the ranked evidence.",
                "Editor agent: Review for coherence, coverage, and citation completeness; request fixes.",
                final_required,
            ]
        # inject/replace first two if missing or out of order
        steps_list = [s for s in steps_list if isinstance(s, str)]
        if not steps_list or steps_list[0] != required_first:
            steps_list = [required_first] + steps_list
        if len(steps_list) < 2 or steps_list[1] != required_second:
            # remove any generic arxiv step that is not tied to Tavily results
            steps_list = (
                [steps_list[0]]
                + [required_second]
                + [
                    s
                    for s in steps_list[1:]
                    if "arXiv" not in s or "For each collected item" in s
                ]
            )
        # ensure final step requirement present
        if final_required not in steps_list:
            steps_list.append(final_required)
        # cap to 7
        return steps_list[:7]

    steps = _ensure_contract(steps)
    return steps

In [19]:
test_topics = ["Recent advances in large language models"] 
# Test with the first topic (or modify to test others)
topic = test_topics[0]
print("🔄 Generating plan...")
steps = planner_agent(topic)
print(f"✅ Plan generated successfully:")
print(steps)

🔄 Generating plan...
✅ Plan generated successfully:
['Research agent: Use Tavily to perform a broad web search and collect top relevant items (title, authors, year, venue/source, URL, DOI if available).', 'Research agent: For each collected item, search on arXiv to find matching preprints/versions and record arXiv URLs (if they exist).', 'Research agent: Evaluate and annotate each source with concise summaries of contributions, methodologies, and relevance to recent large language model advances.', 'Research agent: Rank the annotated sources by novelty, impact, and recency to select top candidates for detailed coverage.', 'Writer agent: Draft detailed section outlines and initial write-ups for the selected advances, incorporating methodology, key results, and future directions.', 'Editor agent: Review and refine the drafted sections for clarity, coherence, completeness, and proper inline citation placeholders.', 'Writer agent: Generate a comprehensive Markdown report that uses all find

In [ ]:

task_progress = {}
task_id = str(uuid.uuid4())
task_progress[task_id] = {"steps": []}
for step_title in steps:
    task_progress[task_id]["steps"].append(
        {
            "title": step_title,               # Human-readable step name
            "status": "pending",               # pending | running | done | error
            "description": "Awaiting execution",
            "substeps": [],                     # Filled with agent calls / outputs
        }
    )
task_progress

{'300954a7-b931-465c-8456-d65e1b1400b9': {'steps': [{'title': 'Research agent: Use Tavily to perform a broad web search and collect top relevant items (title, authors, year, venue/source, URL, DOI if available).',
    'status': 'pending',
    'description': 'Awaiting execution',
    'substeps': []},
   {'title': 'Research agent: For each collected item, search on arXiv to find matching preprints/versions and record arXiv URLs (if they exist).',
    'status': 'pending',
    'description': 'Awaiting execution',
    'substeps': []},
   {'title': 'Research agent: Evaluate and annotate each source with concise summaries of contributions, methodologies, and relevance to recent large language model advances.',
    'status': 'pending',
    'description': 'Awaiting execution',
    'substeps': []},
   {'title': 'Research agent: Rank the annotated sources by novelty, impact, and recency to select top candidates for detailed coverage.',
    'status': 'pending',
    'description': 'Awaiting executi